# Notebook 08: torch.compile

## Why This Matters

`torch.compile` (introduced in PyTorch 2.0) is the most impactful single-line change you can make to a training loop. On NVIDIA GPUs it typically provides 1.5-3x speedups for transformer training, and 1.2-2x for CNNs, with zero model code changes. Understanding what it does -- and importantly, what breaks it (graph breaks due to data-dependent control flow) -- is essential for any production training system. Engineers who apply it blindly without understanding graph breaks waste hours debugging why compile made things slower.

## Background

Eager mode PyTorch executes one operation at a time. For each kernel, Python dispatches to a C++ function, which launches a CUDA kernel, and the CPU moves on to the next operation. This creates a pipeline of thousands of small kernel launches, each with overhead. For small kernels (activations, normalisation, attention masks), the launch overhead can dominate the actual compute.

`torch.compile` addresses this through a three-stage compiler stack:
1. **TorchDynamo**: intercepts Python bytecode, traces operations into an FX graph
2. **AOTAutograd**: captures both forward and backward graphs ahead-of-time
3. **TorchInductor**: generates fused kernels (Triton for GPU, C++/OpenMP for CPU)

The result: many small kernels are fused into fewer, larger kernels, and kernel launch overhead is amortised.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import os

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

# torch.compile requires PyTorch >= 2.0
major, minor = [int(x) for x in torch.__version__.split(".")[:2]]
compile_available = major >= 2
print(f"torch.compile available: {compile_available}")
print("Ready.")

## 1. Eager Mode Overhead

In eager mode, every operation is a separate kernel launch. The Python interpreter, dispatch logic, and CUDA runtime overhead accumulate. For a transformer with attention, LayerNorm, and GELU, this can mean thousands of kernel launches per training step -- each adding microseconds that sum to meaningful wall-clock overhead.

In [ ]:
# Measure eager mode kernel launch overhead
def attention_like_op(x, W_q, W_k, W_v, W_o):
    B, T, D = x.shape
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v
    scale = D ** -0.5
    attn = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)
    out = attn @ V
    return out @ W_o

B, T, D = 8, 64, 128
x = torch.randn(B, T, D, device=device)
Wq = nn.Parameter(torch.randn(D, D, device=device))
Wk = nn.Parameter(torch.randn(D, D, device=device))
Wv = nn.Parameter(torch.randn(D, D, device=device))
Wo = nn.Parameter(torch.randn(D, D, device=device))

# Warmup
for _ in range(5):
    _ = attention_like_op(x, Wq, Wk, Wv, Wo)

# Benchmark eager
N_RUNS = 100
if device == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N_RUNS):
    out = attention_like_op(x, Wq, Wk, Wv, Wo)
if device == "cuda":
    torch.cuda.synchronize()
eager_time = (time.perf_counter() - t0) / N_RUNS * 1000
print(f"Eager mode:   {eager_time:.3f} ms/call")

# Compile and benchmark
if compile_available:
    compiled_fn = torch.compile(attention_like_op)
    # warmup (first call compiles)
    print("Compiling (first call may take 10-30s)...")
    for _ in range(3):
        _ = compiled_fn(x, Wq, Wk, Wv, Wo)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        out = compiled_fn(x, Wq, Wk, Wv, Wo)
    if device == "cuda":
        torch.cuda.synchronize()
    compiled_time = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f"Compiled mode:{compiled_time:.3f} ms/call")
    speedup = eager_time / compiled_time
    print(f"Speedup: {speedup:.2f}x")
else:
    print("torch.compile not available on this PyTorch version")

## 2. Compile Modes

Three modes trade compilation time against runtime performance:

- **`default`**: balanced; enables fusion and some specialisation. Best for most models.
- **`reduce-overhead`**: uses CUDA graphs to reduce CPU-side launch overhead. Best for models with many small, fixed-size ops (transformers with fixed sequence length).
- **`max-autotune`**: tries many kernel configurations and picks the fastest. Compilation takes minutes but achieves maximum throughput. Use only for long production training runs.

In [ ]:
if compile_available:
    model_bench = nn.Sequential(
        nn.Linear(256, 512), nn.GELU(),
        nn.Linear(512, 512), nn.GELU(),
        nn.Linear(512, 256), nn.GELU(),
        nn.Linear(256, 10),
    ).to(device)

    x_bench = torch.randn(64, 256, device=device)

    def run_forward(model, x, n=50):
        for _ in range(3):  # warmup
            _ = model(x)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n):
            _ = model(x)
        if device == "cuda":
            torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n * 1000

    eager_ms = run_forward(model_bench, x_bench)
    print(f"Eager:          {eager_ms:.3f} ms")

    for mode in ["default", "reduce-overhead"]:
        try:
            compiled = torch.compile(model_bench, mode=mode)
            ms = run_forward(compiled, x_bench)
            print(f"compile({mode:20s}): {ms:.3f} ms  ({eager_ms/ms:.2f}x)")
        except Exception as e:
            print(f"compile({mode}): {e}")
else:
    print("Skipping compile mode benchmark -- torch.compile not available")

## 3. Graph Breaks -- What Breaks Compilation

A graph break occurs when TorchDynamo cannot trace through a Python construct and must fall back to eager execution. Graph breaks are the primary source of `torch.compile` not delivering expected speedups. They cause the model to be compiled in fragments, with Python interpreter overhead between fragments.

Common causes of graph breaks:
- Data-dependent control flow: `if tensor.item() > 0:` (value not known at compile time)
- Unsupported Python builtins or third-party library calls
- In-place operations on tensors with aliases
- Dynamic shapes when `dynamic=False` (the default)

In [ ]:
import os

# Demonstrate a graph break with data-dependent control flow
class ModelWithBreak(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 8)

    def forward(self, x):
        x = self.fc1(x)
        # GRAPH BREAK: .item() forces a Python value, breaks the trace
        if x.max().item() > 5.0:
            x = x * 0.1
        return self.fc2(x)

class ModelWithoutBreak(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 8)

    def forward(self, x):
        x = self.fc1(x)
        # FIX: use tensor operations instead of Python conditionals
        x = torch.where(x > 5.0, x * 0.1, x)  # stays in the graph
        return self.fc2(x)

x_test = torch.randn(8, 16)
m_break = ModelWithBreak()
m_clean = ModelWithoutBreak()

# Test forward pass works on both
out_break = m_break(x_test)
out_clean = m_clean(x_test)
print(f"Both models produce output: {out_break.shape}, {out_clean.shape}")

if compile_available:
    # Enable graph break logging
    os.environ["TORCH_LOGS"] = "graph_breaks"
    print("\nCompiling model WITH graph break (watch for warnings):")
    try:
        compiled_break = torch.compile(m_break)
        _ = compiled_break(x_test)
        print("Compiled (with graph break -- check warnings above)")
    except Exception as e:
        print(f"Error: {e}")
    os.environ.pop("TORCH_LOGS", None)

    print("\nCompiling model WITHOUT graph break:")
    compiled_clean = torch.compile(m_clean)
    _ = compiled_clean(x_test)
    print("Compiled cleanly.")
else:
    print("torch.compile not available -- pattern demonstrated above")

## 4. dynamic=True for Variable Batch Sizes

By default, `torch.compile` specialises (recompiles) for each unique input shape. A model that receives batches of size 1, 16, 32, and 64 will be compiled four times. `dynamic=True` enables dynamic shapes: the compiler generates code that works for any shape within a range, recompiling only when shapes change in ways it has not seen.

In [ ]:
if compile_available:
    model_dyn = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10)).to(device)

    # static compile: recompiles for each new batch size
    compiled_static = torch.compile(model_dyn, dynamic=False)
    # dynamic compile: handles variable batch sizes without recompilation
    compiled_dynamic = torch.compile(model_dyn, dynamic=True)

    print("Static compile -- multiple batch sizes (may trigger recompilation):")
    for bs in [4, 8, 16, 32]:
        x = torch.randn(bs, 32, device=device)
        out = compiled_static(x)
        print(f"  batch={bs}: output {out.shape}")

    print("\nDynamic compile -- no recompilation for new batch sizes:")
    for bs in [4, 8, 16, 32]:
        x = torch.randn(bs, 32, device=device)
        out = compiled_dynamic(x)
        print(f"  batch={bs}: output {out.shape}")
else:
    print("torch.compile not available -- dynamic=True example shown conceptually")
    print("dynamic=True: compile once, handle any batch size in [1, 2^31)")

## 5. Applying torch.compile to a Training Loop

The recommended pattern is to compile the model before the training loop. The first forward pass triggers compilation (warm-up); subsequent passes use the compiled code. You can also compile just the forward function or a custom training step function.

In [ ]:
torch.manual_seed(42)

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(32, 128), nn.GELU(),
            nn.Linear(128, 128), nn.GELU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

model_compile = SmallMLP().to(device)

if compile_available:
    model_compile = torch.compile(model_compile, mode="default")
    print("Model compiled.")
else:
    print("Skipping compile -- running in eager mode.")

optimizer_c = torch.optim.AdamW(
    # compiled model still exposes .parameters()
    [p for p in model_compile.parameters()],
    lr=1e-3
)
criterion = nn.CrossEntropyLoss()

X_c = torch.randn(128, 32, device=device)
y_c = torch.randint(0, 10, (128,), device=device)

print("Training (first step triggers compilation warmup):")
for step in range(5):
    t0 = time.perf_counter()
    optimizer_c.zero_grad(set_to_none=True)
    with torch.amp.autocast(
        device_type="cuda" if device == "cuda" else "cpu",
        dtype=torch.bfloat16
    ):
        loss = criterion(model_compile(X_c), y_c)
    loss.backward()
    optimizer_c.step()
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"  step {step}: loss={loss.item():.4f}  time={elapsed:.1f}ms")
print("\nNote: step 0 is slow (compile warmup); subsequent steps are fast.")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| Eager mode overhead | Thousands of kernel launches per step; launch overhead accumulates |
| TorchDynamo | Traces Python bytecode into FX graph; stops at graph breaks |
| AOTAutograd | Captures forward + backward as one graph for joint optimisation |
| TorchInductor | Generates fused Triton (GPU) or C++ (CPU) kernels |
| Graph break | Python .item(), unsupported ops, or data-dependent branching stops tracing |
| dynamic=True | Handles variable shapes without recompilation |
| Compile modes | default (best all-around), reduce-overhead (small ops), max-autotune (long runs) |

### Common Pitfalls
- Using `.item()` inside a compiled forward -- causes graph break, kills speedup
- Expecting speedup on first call -- first call is the compilation; warmup is mandatory
- Not using `dynamic=True` with variable batch sizes -- recompile on every new shape
- Compiling with `max-autotune` for short runs -- compile time exceeds training time
- Applying compile before moving model to device -- TorchInductor needs to know the device

### Next: Notebook 09 -- Memory Profiling and Debugging